In [6]:
import pandas as pd
import numpy as np

clients = pd.read_csv("../data/clean/clients_clean.csv")

print(clients["est_revenu_6m"].value_counts(normalize=True).round(3))
clients.head(5)

est_revenu_6m
non    0.589
oui    0.411
Name: proportion, dtype: float64


,client_id,boutique_principale,anciennete_mois,nb_achats_12m,panier_moyen_eur,retours,demandes_sav,carte_fidelite,canal_prefere,pays_residence,montant_prochain_achat_eur,est_revenu_6m,montant_total_eur,ville,surface_m2,vendeurs,ouverte_depuis_mois,pays_boutique
0,CT80059,BT15,4,4,1468.0,0,0,aucune,en ligne,USA,725.0,oui,5872.0,Séoul,483,8,236,Corée
1,CT80578,BT28,80,3,1235.0,1,1,argent,boutique,France,1479.0,oui,3705.0,Cannes,267,14,182,France
2,CT80256,BT27,43,1,1814.0,0,2,or,boutique,France,2307.0,oui,1814.0,Dubaï,519,12,209,EAU
3,CT80098,BT16,1,1,1795.0,0,0,or,boutique,Italie,1820.0,non,1795.0,Paris,139,21,266,France
4,CT80362,BT31,51,3,2022.0,0,0,argent,boutique,Chine,1201.0,oui,6066.0,Dubaï,108,17,73,EAU


In [2]:
clients.groupby("est_revenu_6m").mean(numeric_only=True).round(1)

,anciennete_mois,nb_achats_12m,panier_moyen_eur,retours,demandes_sav,montant_prochain_achat_eur,montant_total_eur,surface_m2,vendeurs,ouverte_depuis_mois
est_revenu_6m,,,,,,,,,,
non,28.5,3.2,2435.5,0.5,0.4,1841.6,9779.7,292.3,15.2,174.9
oui,34.1,4.4,2940.2,0.5,0.4,2158.5,16816.1,289.5,15.5,177.0


In [15]:
clients["touriste"] = (clients["pays_residence"] != clients["pays_boutique"]).astype(int)


In [16]:
clients.head(5)

,client_id,boutique_principale,anciennete_mois,nb_achats_12m,panier_moyen_eur,retours,demandes_sav,carte_fidelite,canal_prefere,pays_residence,montant_prochain_achat_eur,est_revenu_6m,montant_total_eur,ville,surface_m2,vendeurs,ouverte_depuis_mois,pays_boutique,touriste
0,CT80059,BT15,4,4,1468.0,0,0,aucune,en ligne,USA,725.0,oui,5872.0,Séoul,483,8,236,Corée,1
1,CT80578,BT28,80,3,1235.0,1,1,argent,boutique,France,1479.0,oui,3705.0,Cannes,267,14,182,France,0
2,CT80256,BT27,43,1,1814.0,0,2,or,boutique,France,2307.0,oui,1814.0,Dubaï,519,12,209,EAU,1
3,CT80098,BT16,1,1,1795.0,0,0,or,boutique,Italie,1820.0,non,1795.0,Paris,139,21,266,France,1
4,CT80362,BT31,51,3,2022.0,0,0,argent,boutique,Chine,1201.0,oui,6066.0,Dubaï,108,17,73,EAU,1


In [17]:
print(clients["touriste"].value_counts())

touriste
1    549
0    151
Name: count, dtype: int64


In [18]:
pd.crosstab(clients["canal_prefere"], clients["est_revenu_6m"], normalize="index").round(2)

est_revenu_6m,non,oui
canal_prefere,,
boutique,0.55,0.45
en ligne,0.63,0.37
mixte,0.65,0.35


In [19]:
pd.crosstab(clients["carte_fidelite"], clients["est_revenu_6m"], normalize="index").round(2)

est_revenu_6m,non,oui
carte_fidelite,,
argent,0.57,0.43
aucune,0.61,0.39
inconnue,0.52,0.48
or,0.59,0.41


In [20]:
pd.crosstab(clients["touriste"], clients["est_revenu_6m"], normalize="index").round(2)

est_revenu_6m,non,oui
touriste,,
0,0.60,0.40
1,0.58,0.42


In [22]:
from scipy.stats import chi2_contingency

for col in ["canal_prefere", "carte_fidelite", "touriste"]:
    tableau = pd.crosstab(clients[col], clients["est_revenu_6m"])
    chi2, p, dof, attendu = chi2_contingency(tableau)
    print(f"{col} : p-value = {p:.4f}")

canal_prefere : p-value = 0.0682
carte_fidelite : p-value = 0.7152
touriste : p-value = 0.7614


In [23]:
from scipy.stats import mannwhitneyu

revenus = clients[clients["est_revenu_6m"] == "oui"]
partis = clients[clients["est_revenu_6m"] == "non"]

for col in ["nb_achats_12m", "panier_moyen_eur", "anciennete_mois", "retours", "demandes_sav", "montant_total_eur"]:
    stat, p = mannwhitneyu(revenus[col], partis[col])
    print(f"{col} : p-value = {p:.4f}")

nb_achats_12m : p-value = 0.0000
panier_moyen_eur : p-value = 0.0009
anciennete_mois : p-value = 0.0026
retours : p-value = 0.9346
demandes_sav : p-value = 0.3763
montant_total_eur : p-value = 0.0000


## Conclusion de l'exploration (Q5)

**Ce qui distingue les clients qui partent :**

| Variable | Revenus | Partis | p-value | Lien ? |
|---|---|---|---|---|
| Nb d'achats (12 mois) | 4,4 | 3,2 | < 0,001 | ✅ très fort |
| Montant total (€) | 16 816 | 9 780 | < 0,001 | ✅ très fort |
| Panier moyen (€) | 2 940 | 2 436 | 0,001 | ✅ fort |
| Ancienneté (mois) | 34,1 | 28,5 | 0,003 | ✅ fort |
| Demandes SAV | 0,4 | 0,4 | 0,38 | ❌ aucun |
| Retours | 0,5 | 0,5 | 0,93 | ❌ aucun |
| Canal (boutique 45 % / en ligne 37 % / mixte 35 %) | | | 0,07 | ⚠️ tendance non confirmée |
| Carte de fidélité | | | 0,72 | ❌ aucun |
| Touriste | | | 0,76 | ❌ aucun |

Tests utilisés : Mann-Whitney pour les variables numériques (pas d'hypothèse de loi normale, montants très dispersés), khi-deux pour les variables catégorielles. Seuil : 5 %.

**À retenir :**
- Les clients qui partent sont ceux qui **achètent peu** : moins souvent, pour de plus petits montants, et depuis moins longtemps.
- Un client **mécontent** (retours, SAV) ne part pas plus qu'un autre : c'est le **désengagement** qui annonce le départ.
- La carte **or** ne retient pas mieux que l'absence de carte : le programme de fidélité ne remplit pas son rôle.
- Être **touriste** n'a aucun effet sur le retour.

**Pour le modèle :** `montant_total_eur` = `nb_achats_12m` × `panier_moyen_eur`. Garder les trois créerait de la colinéarité.